In [1]:
import pandas as pd
import numpy as np


In [2]:
FEATURE_PATH = "../Data/Feature_Data"

enrol_feat = pd.read_csv(
    f"{FEATURE_PATH}/feature_enrolment.csv",
    parse_dates=["date"]
)

demo_bio_feat = pd.read_csv(
    f"{FEATURE_PATH}/feature_demo_bio_combined.csv",
    parse_dates=["date"]
)


In [3]:
sort_keys = ["state_clean", "district_clean", "pincode", "date"]

enrol_feat = enrol_feat.sort_values(sort_keys)
demo_bio_feat = demo_bio_feat.sort_values(sort_keys)


In [4]:
enrol_feat["enrolment_dod_change"] = (
    enrol_feat
    .groupby(["state_clean", "district_clean", "pincode"])["total_enrolments"]
    .diff()
)

enrol_feat["enrolment_dod_pct_change"] = (
    enrol_feat
    .groupby(["state_clean", "district_clean", "pincode"])["total_enrolments"]
    .pct_change()
)


In [5]:
enrol_feat["enrolment_7day_mean"] = (
    enrol_feat
    .groupby(["state_clean", "district_clean", "pincode"])["total_enrolments"]
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

enrol_feat["enrolment_7day_std"] = (
    enrol_feat
    .groupby(["state_clean", "district_clean", "pincode"])["total_enrolments"]
    .transform(lambda x: x.rolling(7, min_periods=1).std())
)


In [6]:
enrol_feat["enrolment_volatility"] = np.where(
    enrol_feat["enrolment_7day_mean"] > 0,
    enrol_feat["enrolment_7day_std"] / enrol_feat["enrolment_7day_mean"],
    0
)


In [7]:
enrol_feat["sudden_surge_flag"] = (
    enrol_feat["enrolment_dod_pct_change"] > 0.5
).astype(int)

enrol_feat["sudden_drop_flag"] = (
    enrol_feat["enrolment_dod_pct_change"] < -0.5
).astype(int)


In [8]:
demo_bio_feat["bio_demo_stress_ratio"] = np.where(
    demo_bio_feat["total_demographic_updates"] > 0,
    demo_bio_feat["total_biometric_updates"] /
    demo_bio_feat["total_demographic_updates"],
    0
)


In [9]:
demo_bio_feat["high_biometric_stress_flag"] = (
    demo_bio_feat["bio_demo_stress_ratio"] > 1.5
).astype(int)


In [10]:
import os

ADV_FEATURE_PATH = "../Data/Advanced_Feature_Data"
os.makedirs(ADV_FEATURE_PATH, exist_ok=True)


In [11]:
enrol_feat.to_csv(
    f"{ADV_FEATURE_PATH}/advanced_feature_enrolment.csv",
    index=False
)

demo_bio_feat.to_csv(
    f"{ADV_FEATURE_PATH}/advanced_feature_demo_bio.csv",
    index=False
)


In [12]:
import os
os.listdir("../Data/Advanced_Feature_Data")


['advanced_feature_demo_bio.csv', 'advanced_feature_enrolment.csv']

In [6]:
import pandas as pd


In [12]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


In [13]:
ADV_FEATURE_PATH = "../Data/Advanced_Feature_Data"


In [14]:
adv_enrol_df = pd.read_csv(
    f"{ADV_FEATURE_PATH}/advanced_feature_enrolment.csv",
    parse_dates=["date"]
)

adv_demo_bio_df = pd.read_csv(
    f"{ADV_FEATURE_PATH}/advanced_feature_demo_bio.csv",
    parse_dates=["date"]
)


In [15]:
print("Advanced Enrolment Features — HEAD")
print(adv_enrol_df.head())


Advanced Enrolment Features — HEAD
        date                  state_clean district_clean  pincode  age_0_5  age_5_17  age_18_greater  year  month  \
0 2025-01-09  Andaman and Nicobar Islands       andamans   744101        0         1               0  2025      1   
1 2025-04-09  Andaman and Nicobar Islands       andamans   744101        1         0               0  2025      4   
2 2025-04-09  Andaman and Nicobar Islands       andamans   744103        1         0               0  2025      4   
3 2025-09-09  Andaman and Nicobar Islands       andamans   744103        0         1               0  2025      9   
4 2025-09-11  Andaman and Nicobar Islands       andamans   744103        1         0               0  2025      9   

   quarter  week_of_year  day_of_week  is_weekend  total_enrolments  child_enrolments  adult_ratio  child_share  \
0        1             2            3           0                 1                 1          0.0          1.0   
1        2            15        

In [16]:
print("\nAdvanced Enrolment Features — DESCRIBE")
print(adv_enrol_df.describe())



Advanced Enrolment Features — DESCRIBE
                                date        pincode        age_0_5       age_5_17  age_18_greater      year  \
count                         320114  320114.000000  320114.000000  320114.000000   320114.000000  320114.0   
mean   2025-06-29 23:50:16.737787392  516885.658884       4.881258       2.913874        0.356698    2025.0   
min              2025-01-04 00:00:00  110001.000000       0.000000       0.000000        0.000000    2025.0   
25%              2025-03-11 00:00:00  362265.000000       1.000000       0.000000        0.000000    2025.0   
50%              2025-07-11 00:00:00  516115.000000       2.000000       0.000000        0.000000    2025.0   
75%              2025-10-09 00:00:00  700024.000000       3.000000       1.000000        0.000000    2025.0   
max              2025-12-11 00:00:00  855456.000000    2688.000000    1812.000000      855.000000    2025.0   
std                              NaN  205823.283120      30.305184      

In [17]:
print("\nAdvanced Demographic + Biometric Features — DESCRIBE")
print(adv_demo_bio_df.describe())



Advanced Demographic + Biometric Features — DESCRIBE
                                date        pincode  demo_age_5_17   demo_age_17_  year_demo     month_demo  \
count                         786539  786539.000000  786539.000000  786539.000000   786539.0  786539.000000   
mean   2025-06-23 16:35:28.604429568  526585.983482       4.124136      36.413060     2025.0       6.421521   
min              2025-01-03 00:00:00  110001.000000       0.000000       0.000000     2025.0       1.000000   
25%              2025-03-12 00:00:00  391774.000000       0.000000       3.000000     2025.0       3.000000   
50%              2025-06-12 00:00:00  524305.000000       1.000000       8.000000     2025.0       6.000000   
75%              2025-10-09 00:00:00  695501.000000       3.000000      20.000000     2025.0      10.000000   
max              2025-12-12 00:00:00  855456.000000    2690.000000   16166.000000     2025.0      12.000000   
std                              NaN  200002.856704      2